In [50]:
import os
import sys
import yaml
from collections import namedtuple
import numpy as np

home_path = "/home/fc-3auid-3af522edce-2ddb4d-2d4b8c-2d8500-2df5376270c3e0"
module_path = os.path.join(home_path, "gitprojects/gref4hsi")  # Replace with the actual path
sys.path.append(module_path)

from gref4hsi.utils.config_utils import prepend_data_dir_to_relative_paths, customize_config


# We begin by defining all the stuff that process_hyperspectral does such as the config file:
def define_processing_vars(config_yaml, specim_mission_folder, geoid_path, config_template_path, lab_calibration_path, fast_mode = False):
# Read flight-specific yaml file
    with open(config_yaml, 'r') as file:  
        config_data = yaml.safe_load(file)
    
    
    # assigning the arguments to variables for simple backwards compatibility
    SPECIM_MISSION_FOLDER = specim_mission_folder
    EPSG_CODE = config_data['mission_epsg']
    RESOLUTION_ORTHOMOSAIC = config_data['resolution_orthomosaic']
    CALIBRATION_DIRECTORY = lab_calibration_path
    
    
    dem_fold = os.path.join(specim_mission_folder, "dem")

    if not os.path.exists(dem_fold):
        print('DEM folder does not exist so Geoid is used as terrain instead')
        TERRAIN_TYPE = "geoid"
    else:
        if not os.listdir(dem_fold):
            #print(f"The folder '{dem_fold}' is empty so Geoid is used as terrain instead.")
            TERRAIN_TYPE = "geoid"
        else:
            # If there is a folder and it is not empty
            # Find the only file that is there
            files = [f for f in os.listdir(dem_fold) if f not in ('.', '..')]
            DEM_PATH = os.path.join(dem_fold, files[0])
            #print(f"The file '{DEM_PATH}' is used as terrain.")
            TERRAIN_TYPE = "dem_file"
    
    
    # Do coregistration if there is an orthomosaic to compare under "orthomosaic"
    #do_coreg = True
    ortho_ref_fold = os.path.join(specim_mission_folder, "orthomosaic")
    do_coreg = False
    if not os.path.exists(ortho_ref_fold):
        print('Coregistration is not done, as there was no reference orthomosaic')
        
    else:
        if not os.listdir(ortho_ref_fold):
            print('Coregistration is not done, as there was no reference orthomosaic')
        else:
            # If there is a folder and it is not empty
            # Find the only file that is there
            ortho_ref_file = [f for f in os.listdir(ortho_ref_fold) if f not in ('.', '..')][0]
            do_coreg = True
            print(f"The file '{ortho_ref_file}' is used as as reference orthomosaic.")
            
    
    
    GEOID_PATH = geoid_path

    # Settings associated with preprocessing of data from Specim Proprietary data to pipeline-compatible data
    SettingsPreprocess = namedtuple('SettingsPreprocessing', ['dtype_datacube', 
                                                                            'lines_per_chunk', 
                                                                            'specim_raw_mission_dir',
                                                                            'cal_dir',
                                                                            'reformatted_missions_dir',
                                                                            'rotation_matrix_hsi_to_body',
                                                                            'translation_body_to_hsi',
                                                                            'config_file_name'])

    config_specim_preprocess = SettingsPreprocess(dtype_datacube = np.float32, # The data type for the datacube
                                lines_per_chunk = 2000,  # Raw datacube is chunked into this many lines. GB_per_chunk = lines_per_chunk*n_pixels*n_bands*4 bytes
                                specim_raw_mission_dir = SPECIM_MISSION_FOLDER, # Folder containing several mission
                                cal_dir = CALIBRATION_DIRECTORY,  # Calibration directory holding all calibrations at all binning levels
                                reformatted_missions_dir = os.path.join(SPECIM_MISSION_FOLDER, 'processed'), # The fill value for empty cells (select values not occcuring in cube or ancillary data)
                                rotation_matrix_hsi_to_body = np.array([[0, 1, 0],
                                                                        [-1, 0, 0],
                                                                        [0, 0, 1]]), # Rotation matrix R rotating so that vec_body = R*vec_hsi.
                                translation_body_to_hsi = np.array([0, 0, 0]), # Translation t so that vec_body_to_object = vec_hsi_to_object + t
                                # For large files, RAM issues could be a concern. For rectified files exeeding this size, data is written chunk-wize to a memory map.
                                config_file_name = 'configuration.ini')



    # Where to place the config
    DATA_DIR = config_specim_preprocess.reformatted_missions_dir
    config_file_mission = os.path.join(DATA_DIR, 'configuration.ini')


    # Set the data directory for the mission, and create empty folder structure
    prepend_data_dir_to_relative_paths(config_path=config_template_path, DATA_DIR=DATA_DIR)

    # Non-default settings
    custom_config = {'General':
                        {'mission_dir': DATA_DIR,
                        'model_export_type': TERRAIN_TYPE, # Ray trace onto geoid
                        'max_ray_length': 150}, # Max distance in meters from spectral imager to seafloor. Specim does not fly higher

                    'Coordinate Reference Systems':
                        {'proj_epsg' : EPSG_CODE, # The projected CRS UTM 32, common on mainland norway
                        'geocsc_epsg_export' : 4978, # 3D cartesian system for earth consistent with GPS frame (but inconsistent with eurasian techtonic plate)
                        'dem_epsg' : EPSG_CODE, # (Optional) If you have a DEM this can be used
                        'pos_epsg_orig' : 4978}, # The CRS of the positioning data we deliver to the georeferencing

                    'Orthorectification':
                        {'resample_rgb_only': False, # True can be good choice for speed during DEV
                         'resample_ancillary': True,
                        'resolutionhyperspectralmosaic': RESOLUTION_ORTHOMOSAIC, # Resolution in m
                        'raster_transform_method': 'north_east'}, # North-east oriented rasters.
                    
                    'HDF.raw_nav': {
                        'rotation_reference_type' : 'eul_ZYX', # The vehicle orientations are given in Yaw, Pitch, Roll from the NAV system
                        'is_global_rot' : False, # The vehicles orientations from NAV system are Yaw, Pitch, Roll
                        'eul_is_degrees' : True}, # And given in degrees
                    'Absolute Paths': {
                        'geoid_path' : GEOID_PATH,
                        'orthomosaic_reference_folder' : os.path.join(specim_mission_folder, "orthomosaic"),
                        'ref_ortho_reshaped' : os.path.join(DATA_DIR, "Intermediate", "RefOrthoResampled"),
                        'ref_gcp_path' : os.path.join(DATA_DIR, "Intermediate", "gcp.csv"),
                        'calib_file_coreg' : os.path.join(DATA_DIR, "Output", "HSI_coreg.xml"),
                        # (above) The georeferencing allows processing using norwegian geoid NN2000 and worldwide EGM2008. Also, use of seafloor terrain models are supported. '
                        # At the moment refractive ray tracing is not implemented, but it could be relatively easy by first ray tracing with geoid+tide, 
                        # and then ray tracing from water
                        #'tide_path' : 'D:/HyperspectralDataAll/HI/2022-08-31-060000-Remoy-Specim/Input/tidevann_nn2000_NMA.txt'
                        },
                    
                    # If coregistration is done, then the data must be stored after processing somewhere
                    'HDF.coregistration': {
                            'position_ecef': 'processed/coreg/position_ecef',
                            'quaternion_ecef' : 'processed/coreg/quaternion_ecef'
                        },
                    # These are the ancillary layers to be orthorectified (can select from all entities in "Georeferencing")
                    'Ancillary': {
                            'position_ecef' : 'processed/nav/position_hsi_ecef',
                            'quaternion_ecef' : 'processed/nav/quaternion_hsi_ecef',
                            'points_ecef_crs' : 'processed/georef/points_ecef_crs',
                            #'points_hsi_crs' : 'processed/georef/point_hsi_frame',
                            #'normals_hsi_crs' : 'processed/georef/normals_hsi_frame',
                            'theta_v' : 'processed/georef/theta_v',
                            'theta_s' : 'processed/georef/theta_s',
                            'phi_v' : 'processed/georef/phi_v',
                            'phi_s' : 'processed/georef/phi_s',
                            #'normals_ned_crs' : 'processed/georef/normals_ned_crs',
                            'unix_time_grid' : 'processed/georef/unix_time_grid', # h5 path
                            'pixel_nr_grid': 'processed/georef/pixel_nr_grid', # h5 path
                            #'frame_nr_grid' : 'processed/georef/frame_nr_grid',
                            #'hsi_tide_gridded' : 'processed/georef/hsi_tide_gridded',
                            'hsi_alts_msl' : 'processed/georef/hsi_alts_msl'
                        }
                    
    }

    if TERRAIN_TYPE == 'geoid':
        custom_config['Absolute Paths']['geoid_path'] = GEOID_PATH
        #'geoid_path' : 'data/world/geoids/egm08_25.gtx'
    elif TERRAIN_TYPE == 'dem_file':
        custom_config['Absolute Paths']['dem_path'] = DEM_PATH

    
    
    if do_coreg:
        # No need to orthorectify the data cube initially when coregistration with RGB composites is done
        custom_config['Orthorectification']['resample_rgb_only'] = True
        
        # Here you can set which camera parameters to optimize
        cam_calibrate_dict = {'calibrate_boresight': False,
                          'calibrate_camera': False,
                          'calibrate_lever_arm': False,
                          'calibrate_cx': False,
                          'calibrate_f': False,
                          'calibrate_k1': False,
                          'calibrate_k2': False,
                          'calibrate_k3': False
                          }

        # Here you can set which time-varying errors to estimate
        calibrate_dict_extr = {'calibrate_pos_x': True,
                          'calibrate_pos_y': True,
                          'calibrate_pos_z': True,
                          'calibrate_roll': False,
                          'calibrate_pitch': False,
                          'calibrate_yaw': True}
        
        coreg_dict = {'calibrate_dict': cam_calibrate_dict,
                      'calibrate_per_transect': True, # Whether to calibrate on each transect seperately (True) or to use an entire set of transects for calibration (False)
                      'calibrate_dict_extr': calibrate_dict_extr,
                      'time_node_spacing': 10, #s (set to really large number to yield single node, constant correction)
                      'hard_threshold_m': 10, # m
                      'pos_err_ref_frame': 'ned', # ['ecef' or 'ned'] The ref frame to estimate position errors in
                      'time_interpolation_method': 'linear',
                      'sigma_param' : np.array([2, 2, 5, 0.1, 0.1, 1]) # north [m], east [m], down [m], roll [deg], pitch [deg], yaw [deg] (is different for RTK/PPK!!!!)
                      }
    else:
        # When no coregistration is done, then resample datacube
        custom_config['Orthorectification']['resample_rgb_only'] = False
    
    # 
    if fast_mode:
        custom_config['Orthorectification']['resample_rgb_only'] = False
        custom_config['Orthorectification']['resolutionhyperspectralmosaic'] = 1


    # Customizes the config file according to settings
    customize_config(config_path=config_file_mission, dict_custom=custom_config)


    config = configparser.ConfigParser()
    config.read(config_file_mission)
    return config, config_specim_preprocess, config_file_mission


In [51]:

specim_mission_folder = "/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi"
config_yaml = os.path.join(specim_mission_folder, "config.seabee.yaml")
geoid_path = os.path.join("/home/notebook/cogs_massimal", "no_kv_HREF2018A_NN2000_EUREF89.tif")
config_template_path = os.path.join("/home/notebook/cogs_massimal", "config_template_path_specim.ini")
lab_calibration_path = ''
afov = np.deg2rad(36.5) # degrees

# Choose the product that you want to georeference
processing_lvl  = "2b" # 
preprocess_lvl_dict = {"0": "0_raw",
                       "1a": "1a_radiance",
                       "2a": "2a_reflectance",
                       "2b": "2b_reflectance_gc"} # key word, subfolder name





config, config_specim_preprocess, config_file = define_processing_vars(config_yaml, 
                       specim_mission_folder, 
                       geoid_path, 
                       config_template_path, 
                       lab_calibration_path, 
                       fast_mode = True)

DEM folder does not exist so Geoid is used as terrain instead
Coregistration is not done, as there was no reference orthomosaic


# 1 Locate the corrected data cubes

In [34]:
import glob
# Patterns for searching data cube files
PATTERN_ENVI = '*.hdr'

# Expects the captured data to reside in capture subfolder
capture_dir = os.path.join(specim_mission_folder,
                           "pre-processed",
                           preprocess_lvl_dict[processing_lvl])

# Path for searching the ENVI files
search_path_envi = os.path.normpath(os.path.join(capture_dir, PATTERN_ENVI))
# Finding all files
envi_hdr_file_paths = glob.glob(search_path_envi)
len(envi_hdr_file_paths)

22

In [41]:
from spectral import envi
# Define metadata
# Read all meta data from header file (currently hard coded, but could be avoided I guess)
class Resonon:
    def __init__(self, envi_hdr_file_path, config):
        """Initialize with an envi HDR"""
        self.spectral_image_obj = envi.open(envi_hdr_file_path)
        # Verbosely written out 
        self.n_lines = int(self.spectral_image_obj.metadata['lines'])
        self.n_bands = int(self.spectral_image_obj.metadata['bands'])
        self.n_pix = int(self.spectral_image_obj.metadata['samples'])
        self.binning_spatial = int(self.spectral_image_obj.metadata['sample binning'])
        self.binning_spectral = int(self.spectral_image_obj.metadata['spectral binning'])
        
        self.file_type = self.spectral_image_obj.metadata['file type']
        self.hdr_offset = int(self.spectral_image_obj.metadata['header offset'])
        self.interleave = self.spectral_image_obj.metadata['interleave']
        self.byte_order = self.spectral_image_obj.metadata['byte order']
        self.t_exp_ms = float(self.spectral_image_obj.metadata['shutter'])
        self.t_exp_unit = self.spectral_image_obj.metadata['shutter units']
        self.fps = float(self.spectral_image_obj.metadata['framerate'])
        self.wlen_unit = self.spectral_image_obj.metadata['wavelength units']
        self.gain = float(self.spectral_image_obj.metadata['gain'])

        self.direction = self.spectral_image_obj.metadata['direction']
        self.flip_radiometric_calibration = self.spectral_image_obj.metadata['flip radiometric calibration']
        self.timestamp = self.spectral_image_obj.metadata['timestamp'] # A single timestamp 

        self.target = self.spectral_image_obj.metadata['target']
        self.wavelengths = self.spectral_image_obj.metadata['wavelength']
        
template_img = Resonon(envi_hdr_file_paths[-1], config) # Using the first image



# 2. Describe the fov of the camera (in terms of a camera model) and the rotation/translation of the camera wrt IMU

In [53]:
from scipy.spatial.transform import Rotation as RotLib
from gref4hsi.utils.geometry_utils import CalibHSI

if lab_calibration_path == '':
    # This means that there is no manufacturer precise calibration for the FOV. Assume a pinhole model:
    # See https://github.com/havardlovas/gref4hsi for info
    width = template_img.n_pix
    cx = width/2
    f = width / (2*np.tan(afov/2))
    k1, k2, k3 = 0, 0, 0
    
else:
    pass

# We define the rotations/translations from the user input:

# User set rotation matrix
R_hsi_body = config_specim_preprocess.rotation_matrix_hsi_to_body

r_zyx = RotLib.from_matrix(R_hsi_body).as_euler('ZYX', degrees=False)

# Euler angle representation (any other would do too)
rotation_z = r_zyx[0]
rotation_y = r_zyx[1]
rotation_x = r_zyx[2]

# Vector from origin of HSI to body origin, expressed in body
# User set
t_hsi_body = config_specim_preprocess.translation_body_to_hsi
translation_x = t_hsi_body[0]
translation_y = t_hsi_body[1]
translation_z = t_hsi_body[2]



param_dict = {'rx':rotation_x,
              'ry':rotation_y,
              'rz':rotation_z,
              'tx':translation_x,
              'ty':translation_y,
              'tz':translation_z,
              'f': f,
              'cx': cx,
              'k1': k1,
              'k2': k2,
              'k3': k3,
              'width': width}

file_name_xml = 'HSI_' + str(template_img.binning_spatial) + 'b.xml'

camera_calib_xml_dir = config['Absolute Paths']['calib_folder'] # Where we put geometric calib files

xml_cal_write_path = os.path.join(camera_calib_xml_dir, file_name_xml)

CalibHSI(file_name_cal_xml= xml_cal_write_path, 
                    mode = 'w', 
                    param_dict = param_dict)

# Set value in config file:
config.set('Relative Paths', 'hsi_calib_path', value = xml_cal_write_path)
# Write the config object 
with open(config_file, 'w') as configfile:
        config.write(configfile)

# 3. Extraction of navigation data

In [64]:
# The processing in massipipe nicely renders the necessary navigation data in a json format

import json
nav_pattern = '*.json'

nav_dir = os.path.join(specim_mission_folder,
                           "pre-processed",
                           'imudata')
search_path_nav = os.path.normpath(os.path.join(nav_dir, nav_pattern))
nav_file_paths = glob.glob(search_path_nav)

print(nav_file_paths[3])
print(envi_hdr_file_paths[3])

example_json = nav_file_paths[-1]


with open(example_json, 'r') as f:
    data = json.load(f)
    
# Example call to get the "time attribute"
print(np.array(data['time']))

/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi/pre-processed/imudata/pre-processed_013_imudata.json
/home/notebook/cogs_massimal/massimal_bodo_saltstraumen_202203121143-small_hsi/pre-processed/2b_reflectance_gc/pre-processed_009_reflectance_gc.bip.hdr
[0.00000000e+00 1.66660000e-02 3.33320000e-02 ... 3.32833700e+01
 3.33000360e+01 3.33099999e+01]
